# Validation protocol for radiomic survival analysis

Runs the whole analysis of the paper: cohort table, step 1 (feature selection and its stability),
repeated nested CV with the two paths, and the negative control. Outputs go to `results/`;
the R scripts in `R/` make the figures from there.

Seeds are saved one by one, so if the run stops just execute the notebook again and finished seeds are loaded.
About 25 min per seed and dataset on a laptop.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from radiomics_protocol import config
from radiomics_protocol.utils import cohort_table, load_radiomics_dataset, prepare_dataset
from radiomics_protocol.pipeline import (
    conditional_frequency, export_summaries, run_repeated_validation, selected_features_table,
    selection_stability, single_split_report, stability_summary, empty_report,
)

SEED = config.DEFAULT_SEED
np.random.seed(SEED)
SEEDS = config.default_seeds()      # 42-51

RESULTS = config.get_results_dir()
TABLES = RESULTS / "tables"
for d in (RESULTS, TABLES, RESULTS / "single_split"):
    d.mkdir(parents=True, exist_ok=True)

if config.FAST_MODE:
    print("RADIOMICS_FAST=1: reduced grids and seeds, smoke test only")
print(ROOT, SEEDS)

/Users/marielatalaveramunoz/Desktop/articulo_radiomica/protocol_rsasc/radiomics-survival-validation-protocol [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [5]:
# step 1: FDR q <= 0.20, then |rho| >= 0.80 redundancy filter (single linkage, keep best univariate C-index)
STEP1 = dict(
    alpha_fdr=config.ALPHA_FDR,
    corr_method=config.CORR_METHOD,
    corr_threshold=config.CORR_THRESHOLD,
    criterion=config.CRITERION,
    ascending=config.ASCENDING,
)

## Data

In [6]:
df_raw, data_path = load_radiomics_dataset()
ds = prepare_dataset(df_raw)
X_raw, y = ds.X, ds.y

print(data_path)
print(f"{ds.n_raw} patients in the file, {len(X_raw)} with complete radiomics")
print(f"{X_raw.shape[1]} features, {int(y['exitus'].sum())} events ({100 * y['exitus'].mean():.1f}%)")

/Users/marielatalaveramunoz/Desktop/articulo_radiomica/protocol_rsasc/radiomics-survival-validation-protocol/data/radiomics_dataset.csv
101 patients in the file, 78 with complete radiomics
535 features, 27 events (34.6%)


## Cohort (table 1)

In [7]:
table1 = cohort_table(ds.full)
table1.to_csv(TABLES / "table1_cohort.csv", sep=config.CSV_SEP)
table1

,Overall (N=78),Events (N=27),Censored (N=51)
"Study cohort, N (%)",78 (100.0%),27 (34.6%),51 (65.4%)
"Sex: F, n (%)",38 (48.7%),16 (59.3%),22 (43.1%)
"Sex: M, n (%)",40 (51.3%),11 (40.7%),29 (56.9%)
"Age (years), median (IQR)",63.0 (52.2–75.0),72.0 (62.0–77.5),58.0 (46.0–73.0)
"Age group <55, n (%)",21 (26.9%),3 (11.1%),18 (35.3%)
"Age group 55-65, n (%)",22 (28.2%),5 (18.5%),17 (33.3%)
"Age group 65-75, n (%)",19 (24.4%),11 (40.7%),8 (15.7%)
"Age group >75, n (%)",16 (20.5%),8 (29.6%),8 (15.7%)
"Overall survival time (days), median (IQR)",2523.0 (1671.0–3937.0),1704.0 (884.5–3494.5),2627.0 (2028.0–4348.5)


## Step 1: feature selection

Reference split: the 70/30 split of seed 42 (the same one used as "single split" below).

In [8]:
ref = single_split_report(X_raw, y, seed=SEED, **STEP1)

print(f"after FDR filter: {ref['n_fdr']}, after redundancy filter: {len(ref['selected'])}")
ref["cox_selected"][["variable", "HR", "HR_lower95", "HR_upper95", "p_raw", "p_adj_fdr", "concordance", "p_schoenfeld"]]

after FDR filter: 41, after redundancy filter: 7


,variable,HR,HR_lower95,HR_upper95,p_raw,p_adj_fdr,concordance,p_schoenfeld
0,Corr_T1_glcm_Idmn_seg3,3.873277,1.688007,8.887569,0.001396,0.074693,0.686901,0.264784
1,Corr_T1_ngtdm_Contrast_seg5,2.068750,1.301233,3.288978,0.002119,0.080072,0.698083,0.954739
2,Corr_T1_shape_MajorAxisLength_seg5,0.334550,0.161806,0.691716,0.003132,0.088190,0.635783,0.021066
3,Corr_T1_shape_Sphericity_seg4,0.224542,0.082651,0.610024,0.003398,0.090894,0.747604,0.368755
4,Corr_T1_glszm_GrayLevelNonUniformity_seg5,0.351619,0.168753,0.732646,0.005262,0.130288,0.664537,0.140443
5,Corr_T1_ngtdm_Contrast_seg4,0.410700,0.214123,0.787747,0.007408,0.132118,0.627796,0.194806
6,Corr_T1_glcm_Correlation_seg5,2.069752,1.225344,3.496058,0.006532,0.132118,0.682109,0.474874


In [9]:
# inputs of the R script for figure 4 (summary statistics only, no patient data)
SS = RESULTS / "single_split"
ref["cox_selected"].to_csv(SS / "rad_sig_cox.csv", sep=config.CSV_SEP, index=False)
ref["spearman"].to_csv(SS / "rad_sig_spearman.csv", sep=config.CSV_SEP)
ref["df_cox"].to_csv(SS / "univariate_cox_all_features.csv", sep=config.CSV_SEP, index=False)

### Stability

Step 1 on 100 random 70/30 splits (StratifiedShuffleSplit on the event, random_state 42).
SF = fraction of repetitions that select the feature, CSF = same but only over repetitions with at least one feature.

In [19]:
freq, sizes, jaccard, sel_sets = selection_stability(
    X_raw, y, n_rep=config.N_STABILITY_REPS, seed=SEED, **STEP1)
freq_cond = conditional_frequency(freq, sizes)

summ = stability_summary(freq, sizes, jaccard, sel_sets)
print(f"selected: {summ['size_mean']:.1f} +- {summ['size_sd']:.1f}, empty in {summ['n_empty']}/{summ['n_rep']}")
print(f"non-empty size: {summ['size_nonempty_mean']:.1f} +- {summ['size_nonempty_sd']:.1f}")
#print(f"jaccard: {summ['jaccard_mean']:.2f} +- {summ['jaccard_sd']:.2f} (non-empty: {summ['jaccard_nonempty_mean']:.2f} +- {summ['jaccard_nonempty_sd']:.2f})")

stab = pd.DataFrame({"SF": freq, "CSF": freq_cond})
stab.to_csv(TABLES / "selection_stability.csv", sep=config.CSV_SEP, index_label="variable")
pd.Series(summ).to_csv(TABLES / "selection_stability_summary.csv", sep=config.CSV_SEP, header=["value"])
stab.head(15)

selected: 6.8 +- 6.8, empty in 28/100
non-empty size: 9.5 +- 6.2


,SF,CSF
Corr_T1_glszm_GrayLevelNonUniformity_seg5,0.53,0.736111
Corr_T1_shape_Sphericity_seg4,0.50,0.694444
Corr_T1_gldm_DependenceEntropy_seg1,0.45,0.625000
Corr_T1_ngtdm_Contrast_seg4,0.41,0.569444
Corr_T1_shape_MajorAxisLength_seg5,0.25,0.347222
Corr_T1_gldm_GrayLevelNonUniformity_seg1,0.25,0.347222
Corr_T1_ngtdm_Contrast_seg5,0.25,0.347222
Corr_T1_glszm_SizeZoneNonUniformity_seg5,0.20,0.277778
Corr_T1_glrlm_GrayLevelNonUniformity_seg5,0.18,0.250000
Corr_T1_gldm_GrayLevelNonUniformity_seg5,0.17,0.236111


### Table 2

In [11]:
table2 = selected_features_table(ref["cox_selected"], freq, freq_cond)
table2.to_csv(TABLES / "table2_selected_features.csv", sep=config.CSV_SEP, index=False)
table2

,compartment,family,feature,variable,HR,HR_lower95,HR_upper95,p_raw,p_adj_fdr,concordance,p_schoenfeld,SF,CSF
0,Intramuscular fat,glcm,Idmn,Corr_T1_glcm_Idmn_seg3,3.873277,1.688007,8.887569,0.001396,0.074693,0.686901,0.264784,0.06,0.083333
1,Skeletal muscle,shape,Sphericity,Corr_T1_shape_Sphericity_seg4,0.224542,0.082651,0.610024,0.003398,0.090894,0.747604,0.368755,0.50,0.694444
2,Skeletal muscle,ngtdm,Contrast,Corr_T1_ngtdm_Contrast_seg4,0.410700,0.214123,0.787747,0.007408,0.132118,0.627796,0.194806,0.41,0.569444
3,Subcutaneous fat,ngtdm,Contrast,Corr_T1_ngtdm_Contrast_seg5,2.068750,1.301233,3.288978,0.002119,0.080072,0.698083,0.954739,0.25,0.347222
4,Subcutaneous fat,shape,MajorAxisLength,Corr_T1_shape_MajorAxisLength_seg5,0.334550,0.161806,0.691716,0.003132,0.088190,0.635783,0.021066,0.25,0.347222
5,Subcutaneous fat,glszm,GrayLevelNonUniformity,Corr_T1_glszm_GrayLevelNonUniformity_seg5,0.351619,0.168753,0.732646,0.005262,0.130288,0.664537,0.140443,0.53,0.736111
6,Subcutaneous fat,glcm,Correlation,Corr_T1_glcm_Correlation_seg5,2.069752,1.225344,3.496058,0.006532,0.132118,0.682109,0.474874,0.07,0.097222


## Repeated nested CV

Ten seeds. For each one: 5-fold nested CV and a single 70/30 split, each with path A (selected features)
and path B (all features), five models. All models run in the same call.

In [12]:
long_real, sel_real = run_repeated_validation(
    X_raw, y, seeds=SEEDS, out_dir=RESULTS / "real", **STEP1)
long_real.shape

[seed 42] ready in 25.5 min
[seed 43] ready in 23.6 min
[seed 44] ready in 24.3 min
[seed 45] ready in 24.5 min
[seed 46] ready in 23.9 min
[seed 47] ready in 23.7 min
[seed 48] ready in 24.2 min
[seed 49] ready in 23.0 min
[seed 50] ready in 23.1 min
[seed 51] ready in 24.9 min


(1800, 10)

## Negative control

Same thing with the features replaced by Gaussian noise.

In [13]:
long_null, sel_null = run_repeated_validation(
    X_raw, y, seeds=SEEDS, null="gaussian", out_dir=RESULTS / "null", **STEP1)
long_null.shape

[seed 42] ready in 20.6 min
[seed 43] ready in 20.4 min
[seed 44] ready in 21.9 min
[seed 45] ready in 21.5 min
[seed 46] ready in 20.8 min
[seed 47] ready in 22.4 min
[seed 48] ready in 24.2 min
[seed 49] ready in 23.2 min
[seed 50] ready in 26.8 min
[seed 51] ready in 22.3 min


(1800, 10)

## Summary tables

In [14]:
tables_real = export_summaries(long_real, sel_real, RESULTS / "real" / "summary")
tables_null = export_summaries(long_null, sel_null, RESULTS / "null" / "summary")
tables_real["delta_summary"]

n_seeds  n_valid  nested_mean  \
path model     metric                                   
A    Cox_LASSO auc           10        9        0.623   
               ci_test       10       10        0.606   
               ibs           10        9        0.192   
     Cox_RIDGE auc           10        9        0.625   
               ci_test       10       10        0.617   
               ibs           10        9        0.189   
     GBM       auc           10        9        0.614   
               ci_test       10       10        0.619   
               ibs           10        9        0.192   
     RSF       auc           10        9        0.647   
               ci_test       10       10        0.643   
               ibs           10        9        0.175   
     SSVM      auc           10        9        0.625   
               ci_test       10       10        0.630   
               ibs           10        0          NaN   
B    Cox_LASSO auc           10        9        0.625   
               ci_test       10       10        0.607   
               ibs           10       10        0.189   
     Cox_RIDGE auc           10        9        0.649   
               ci_test       10       10        0.640   
               ibs           10       10        0.208   
     GBM       auc           10        9        0.580   
               ci_test       10       10        0.586   
               ibs           10       10        0.191   
     RSF       auc           10        9        0.663   
               ci_test       10       10        0.652   
               ibs           10       10        0.181   
     SSVM      auc           10        9        0.579   
               ci_test       10       10        0.582   
               ibs           10        0          NaN   

                        sd_nested_mean_between_seeds  mean_fold_sd  \
path model     metric                                                
A    Cox_LASSO auc                             0.073         0.158   
               ci_test                         0.050         0.135   
               ibs                             0.014         0.028   
     Cox_RIDGE auc                             0.073         0.155   
               ci_test                         0.049         0.134   
               ibs                             0.018         0.040   
     GBM       auc                             0.082         0.116   
               ci_test                         0.055         0.094   
               ibs                             0.024         0.035   
     RSF       auc                             0.073         0.134   
               ci_test                         0.051         0.105   
               ibs                             0.012         0.031   
     SSVM      auc                             0.085         0.129   
               ci_test                         0.053         0.109   
               ibs                               NaN           NaN   
B    Cox_LASSO auc                             0.074         0.139   
               ci_test                         0.037         0.130   
               ibs                             0.008         0.029   
     Cox_RIDGE auc                             0.058         0.147   
               ci_test                         0.037         0.139   
               ibs                             0.027         0.051   
     GBM       auc                             0.056         0.146   
               ci_test                         0.055         0.101   
               ibs                             0.010         0.028   
     RSF       auc                             0.047         0.150   
               ci_test                         0.019         0.134   
               ibs                             0.008         0.032   
     SSVM      auc                             0.064         0.154   
               ci_test                         0.049         0.142   
               ibs           

In [15]:
# outer folds where step 1 kept at least one feature
for name, long in [("radiomic", long_real), ("noise", long_null)]:
    e = empty_report(long).loc["nested"]
    print(f"{name:9s} {100 * (1 - e.n_empty / e.n_splits):.0f}% of {int(e.n_splits)} folds")

radiomic  84% of 50 folds
noise     18% of 50 folds


### Checks

In [16]:
# SSVM under noise should be around 0.5
m = long_null[(long_null.model == "SSVM") & (long_null.metric == "ci_test")]
m.groupby(["scheme", "path"])["value"].agg(["mean", "std"]).round(3)

mean    std
scheme path              
nested A     0.483  0.051
       B     0.545  0.160
single A     0.496  0.011
       B     0.526  0.107

In [17]:
# fits that failed
long_real.groupby(["path", "model"])["failed"].sum().unstack()

model,Cox_LASSO,Cox_RIDGE,GBM,RSF,SSVM
path,,,,,
A,0,0,0,0,0
B,0,0,0,0,0


In [18]:
# to compare with an older run (empty result = same numbers):
# from radiomics_protocol.pipeline import load_run, compare_runs
# long_old, _ = load_run("path/to/old/rep_real")
# compare_runs(long_real, long_old, models=["Cox_RIDGE", "Cox_LASSO", "RSF", "GBM"])